# Lab 2 — Many skills, one router (EP14)

Lab 1 ended with one skill adding input tokens to **every** model call — what about many?
This lab has a library of six skills, and compares the two ways from the theory video

**What you'll do**
1. A library of six skills, and **discovery**: read only each skill's header
2. **Static loading**: every skill, all the time — and count the tokens
3. A **skill router**: the model picks skills from their names and descriptions (dynamic discovery)
4. The whole lifecycle: select → activate → execute → deactivate
5. A vague description makes the router **miss** — then we fix it
6. **Permissions**: users with different roles see different skills

The paper records are teaching data (the arXiv ids, titles, authors and years are real); the model outputs come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §22.2.1–22.2.2, §22.3.4, §22.4, §22.5 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | set up as in Lab 1 (this lab costs less than 0.1 US cents) |
| Packages | `openai`, `python-dotenv`, `pyyaml` (installed in the next cell) |

How to add the key, and why we chose this model, are in Lab 1 — **Colab:** 🔑 Secrets → `OPENROUTER_API_KEY` · **your own machine:** a `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv pyyaml

In [ ]:
import json, os, re
from pathlib import Path
import yaml
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"   # why this model and provider: Lab 1
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0}

In [ ]:
PAPERS = {  # teaching data: real arXiv ids, titles, authors and years; one-line summaries written for this lab
    "2310.08560": {"title": "MemGPT: Towards LLMs as Operating Systems", "authors": "Packer et al.", "year": 2023,
                   "summary": "the LLM pages facts between a small main context and a large external store, like an operating system's virtual memory"},
    "2304.03442": {"title": "Generative Agents: Interactive Simulacra of Human Behavior", "authors": "Park et al.", "year": 2023,
                   "summary": "agents keep a memory stream of observations, score memories by recency, importance and relevance, and write reflections"},
    "2303.11366": {"title": "Reflexion: Language Agents with Verbal Reinforcement Learning", "authors": "Shinn et al.", "year": 2023,
                   "summary": "after a failed try the agent writes a short lesson to its memory and uses it on the next try"},
    "2504.19413": {"title": "Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory", "authors": "Chhikara et al.", "year": 2025,
                   "summary": "an LLM extracts facts from each conversation and decides to add, update or delete them in a long-term store"},
    "2502.12110": {"title": "A-MEM: Agentic Memory for LLM Agents", "authors": "Xu et al.", "year": 2025,
                   "summary": "each memory is a linked note with keywords and tags, and new notes update links to old ones"},
    "2305.16291": {"title": "Voyager: An Open-Ended Embodied Agent with Large Language Models", "authors": "Wang et al.", "year": 2023,
                   "summary": "a Minecraft agent that stores verified code as a growing skill library and reuses it"},
    "2210.03629": {"title": "ReAct: Synergizing Reasoning and Acting in Language Models", "authors": "Yao et al.", "year": 2023,
                   "summary": "the model alternates reasoning steps and tool actions in one loop"},
    "2302.04761": {"title": "Toolformer: Language Models Can Teach Themselves to Use Tools", "authors": "Schick et al.", "year": 2023,
                   "summary": "a model learns by itself where to insert API calls in text"},
}

def search_papers(query: str, max_results: int = 5) -> str:
    words = [w for w in re.findall(r"[a-z]+", query.lower()) if len(w) > 2]
    hits = sorted(((sum(w in (p["title"] + " " + p["summary"]).lower() for w in words), pid)
                   for pid, p in PAPERS.items()), reverse=True)
    return json.dumps([{"id": pid, "title": PAPERS[pid]["title"]} for score, pid in hits[:max_results] if score])

def get_paper(paper_id: str) -> str:
    if paper_id not in PAPERS:
        return json.dumps({"error": f"unknown id {paper_id}; call search_papers first"})
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

FUNCS = {"search_papers": search_papers, "get_paper": get_paper}
SPECS = {
    "search_papers": {"type": "function", "function": {"name": "search_papers",
        "description": "Search the paper library by keywords in the title or summary. Returns ids and titles.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"},
                       "max_results": {"type": "integer"}}, "required": ["query"]}}},
    "get_paper": {"type": "function", "function": {"name": "get_paper",
        "description": "Get one paper's full record (title, authors, year, summary) by its arXiv id.",
        "parameters": {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}}},
}

def run_agent(system, task, tools, max_steps=8):
    """The simple loop from Lab 1; info["input_tokens"] now adds up the input tokens of every call."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    info = {"input_tokens": 0, "calls": 0}
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, messages=msgs,
                                           tools=[SPECS[t] for t in tools] or None)
        COST["chat"] += r.usage.cost
        info["input_tokens"] += r.usage.prompt_tokens
        info["calls"] += 1
        msg = r.choices[0].message
        if not msg.tool_calls:
            return msg.content, info
        msgs.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  🔧 {call.function.name}({args})")
            msgs.append({"role": "tool", "tool_call_id": call.id, "content": FUNCS[call.function.name](**args)})
    return None, info

BASE = "You are a research assistant for university students."

## 1. The skill library, and discovery

| skill | What it does | tools |
|---|---|---|
| `paper-summary` | summarizes papers, a citation on every sentence (from Lab 1) | search, get |
| `paper-quiz` | writes one quiz question per paper | search, get |
| `formatter` | ⚠️ its description is badly written — we come back to it in part 5 | search, get |
| `code-review` | reviews Python code | — |
| `meeting-notes` | turns a meeting into action items | — |
| `grade-submission` | grades against a rubric — **teachers only** | — |

**Discovery** (§22.4, stage 1): read **only the header** of every `SKILL.md` — don't load the bodies yet

In [ ]:
LIBRARY = {
    "paper-summary": """---
name: paper-summary
description: Summarize research papers from the library for students, with a citation on every sentence.
version: 1.0.0
tools: [search_papers, get_paper]
knowledge: [example.md]
---
# Paper summary
Write for first-year students: plain words, no jargon without a short explanation.
## Workflow
1. Call search_papers with the topic.
2. Call get_paper for each paper you will use. Use only what get_paper returns.
3. Compare the papers: what does each one add?
4. Write the answer.
## Guardrails
- Answer with JSON only, no other text: {"papers": ["<id>", ...], "sentences": [{"text": "...", "cite": "<id>"}, ...]}
- Every sentence cites exactly one paper id that you fetched with get_paper.
- At most 120 words in all sentences together.
- Add no claim, number or adjective that the paper record does not contain.
""",
    "paper-quiz": """---
name: paper-quiz
description: Write multiple-choice quiz questions about papers in the library, one question per paper.
version: 1.0.0
tools: [search_papers, get_paper]
---
# Paper quiz
Write quiz questions that check a student understood the main idea of a paper.
## Workflow
1. Find the paper (search_papers) and read its record (get_paper).
2. Write one question about the paper's main idea, from its summary only.
3. Write four choices: one correct, three plausible but wrong.
## Guardrails
- JSON only: {"questions": [{"paper": "<id>", "q": "...", "choices": ["...", "...", "...", "..."], "answer": 1}]}
- The correct answer must follow from the summary; no trick questions.
""",
    "formatter": """---
name: formatter
description: Formats text.
version: 1.0.0
tools: [search_papers, get_paper]
---
# Citation format (APA)
## Workflow
1. Find the paper's arXiv id (search_papers) if you only have its title.
2. Call get_paper with the arXiv id.
3. Write: Authors (Year). Title. arXiv:<id>.
## Guardrails
- Use only the authors, year and title from get_paper; never guess missing parts.
- One line per paper, nothing else.
""",
    "code-review": """---
name: code-review
description: Review Python code for bugs, unclear names and missing error handling.
version: 1.0.0
tools: []
---
# Code review
## Workflow
1. Read the code line by line.
2. List bugs first, then readability, then style.
## Guardrails
- Quote the line you comment on.
- Suggest a fix for every bug.
- Never rewrite the whole program.
""",
    "meeting-notes": """---
name: meeting-notes
description: Turn a meeting transcript into decisions and action items with owners and dates.
version: 1.0.0
tools: []
---
# Meeting notes
## Workflow
1. Read the transcript.
2. List decisions, then action items (owner, task, due date).
## Guardrails
- Only items said in the transcript; mark a missing owner or date as "?".
""",
    "grade-submission": """---
name: grade-submission
description: Grade a student's answer against the course rubric and explain the score.
version: 1.0.0
tools: []
allowed_roles: [teacher]
---
# Grade submission
## Workflow
1. Read the rubric and the answer.
2. Score each rubric line, then add them up.
## Guardrails
- Explain every point taken off.
- Never reveal other students' answers.
""",
}
EXAMPLE = """# Example of a good answer (topic: tool use)
{"papers": ["2210.03629", "2302.04761"],
 "sentences": [
  {"text": "ReAct lets a model think and then act with a tool, again and again in one loop.", "cite": "2210.03629"},
  {"text": "Toolformer goes further: the model learns by itself where an API call helps.", "cite": "2302.04761"}]}
"""
for name, text in LIBRARY.items():
    Path(f"skills/{name}").mkdir(parents=True, exist_ok=True)
    Path(f"skills/{name}/SKILL.md").write_text(text)
Path("skills/paper-summary/example.md").write_text(EXAMPLE)
print(sorted(p.parent.name for p in Path("skills").glob("*/SKILL.md")))

In [ ]:
def read_header(path):
    _, header, body = Path(path).read_text().split("---", 2)
    return yaml.safe_load(header), body.strip()

def discover(root="skills"):
    registry = {}
    for path in sorted(Path(root).glob("*/SKILL.md")):
        meta, _ = read_header(path)
        registry[meta["name"]] = {**meta, "path": path}
    return registry

REGISTRY = discover()
for name, s in REGISTRY.items():
    print(f"{name:17} {s['description']}")

## 2. Static loading: every skill, all the time (§22.2.1)

`activate` now takes **several** skills: it joins all their bodies into the system prompt, and combines all their tools

In [ ]:
def load_skill(name):
    meta, body = read_header(REGISTRY[name]["path"])
    knowledge = "\n\n".join((REGISTRY[name]["path"].parent / f).read_text() for f in meta.get("knowledge", []))
    return {**meta, "instructions": body, "knowledge_text": knowledge}

def activate(skills, base=BASE):
    parts = [base] + [f"# Skill: {s['name']}\n{s['instructions']}\n{s['knowledge_text']}" for s in skills]
    tools = sorted({t for s in skills for t in s["tools"]})
    return "\n\n".join(parts), tools

In [ ]:
TASK = "Summarize 3 papers on agent memory for first-year students."

system, tools = activate([load_skill(n) for n in REGISTRY])
answer, static = run_agent(system, TASK, tools)
print(f"\nstatic: {static['calls']} calls · {static['input_tokens']} input tokens · "
      f"{static['input_tokens'] // static['calls']} per call")

## 3. A skill router: the model picks the skills (§22.2.2)

The router sees only **names + descriptions** and answers with a JSON schema whose `enum` is the skill names (or `"none"`) — it can't answer a name that doesn't exist

In [ ]:
def route(request, names, k=2):
    menu = "\n".join(f"- {n}: {REGISTRY[n]['description']}" for n in names)
    schema = {"type": "object", "additionalProperties": False, "required": ["skills"],
              "properties": {"skills": {"type": "array", "minItems": 1, "maxItems": k,
                                        "items": {"type": "string", "enum": names + ["none"]}}}}
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        response_format={"type": "json_schema", "json_schema": {"name": "route", "strict": True, "schema": schema}},
        messages=[{"role": "system", "content": f"Pick the skills needed for the user's request, at most {k}. "
                                                f"Pick 'none' if no skill fits.\n\nSkills:\n{menu}"},
                  {"role": "user", "content": request}])
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)["skills"], r.usage.prompt_tokens

In [ ]:
for request in ["Summarize 3 papers on agent memory for first-year students.",
                "Write a quiz question about the MemGPT paper.",
                "Is there a bug in: def avg(xs): return sum(xs) / len(xs)",
                "What's the capital of France?"]:
    chosen, tokens = route(request, list(REGISTRY))
    print(f"{str(chosen):22} {tokens:>4} tokens  ← {request}")

## 4. The whole lifecycle (§22.4)

`handle` = discovery (registry) → **selection** (router) → **activation** → **execution** → **deactivation**
(the next request starts from `BASE` again — the skill's context doesn't carry over)

In [ ]:
def handle(request, names=None):
    names = names or list(REGISTRY)
    chosen, router_tokens = route(request, names)
    print(f"🧭 selection: {chosen} ({router_tokens} tokens)")
    if chosen == ["none"]:
        return run_agent(BASE, request, [])
    skills = [load_skill(n) for n in chosen]
    system, tools = activate(skills)
    print(f"🔌 activation: {[s['name'] for s in skills]} · tools {tools}")
    answer, info = run_agent(system, request, tools)
    info["input_tokens"] += router_tokens
    print(f"🧹 deactivation: {chosen} removed — the next request starts from BASE")
    return answer, info

In [ ]:
answer, dynamic = handle(TASK)
print(f"\n{'':10}{'calls':>6}{'input tokens':>14}")
print(f"{'static':10}{static['calls']:>6}{static['input_tokens']:>14}")
print(f"{'dynamic':10}{dynamic['calls'] + 1:>6}{dynamic['input_tokens']:>14}   (router call included)")

## 5. When the router misses: a vague description

The book: *"Routing errors can miss relevant skills"* and *"invest in tool quality"* (§22.2.2, §22.3.4)

In [ ]:
CITE = "How do I cite the MemGPT paper in my report?"
print(route(CITE, list(REGISTRY)))

In [ ]:
path = REGISTRY["formatter"]["path"]
path.write_text(path.read_text().replace(
    "description: Formats text.",
    "description: Format a reference (citation) for a paper in the library in APA style, from its arXiv id or title."))
REGISTRY = discover()

answer, _ = handle(CITE)
print("\n" + answer)

## 6. Permissions: not everyone should get every skill (§22.5)

`allowed_roles` in the header → filter before the router: a student's router **never sees** `grade-submission`

In [ ]:
def visible(role):
    return [n for n, s in REGISTRY.items() if role in s.get("allowed_roles", [role])]

GRADE = "Grade this answer with the rubric: 'MemGPT stores facts in a database.'"
for role in ["student", "teacher"]:
    print(f"{role:8} sees {len(visible(role))} skills → router: {route(GRADE, visible(role))[0]}")

In [ ]:
print(f"cost: ${COST['chat']:.5f}")

## Summary

| | static (every skill) | dynamic (router) |
|---|---|---|
| context | all six bodies, on every model call | names + descriptions to choose + only the chosen bodies |
| model calls | the task only | + one router call |
| risk | wastes context, doesn't scale to hundreds | misses when a description is poor |

- A skill's description is the **only** thing the router sees — `Formats text.` missed, a clear description hit
- Permissions filter **before** the router: a student never sees `grade-submission`

**Exercises**
1. Add a seventh skill, `paper-compare`, that compares two papers — no code changes, just write the folder and `discover()` again
2. Try a request that needs two skills, e.g. *"Summarize the MemGPT paper and give its APA reference"* — how many does the router pick?
3. Replace the router with simple word matching (count words shared with each description) and see where it misses

**Next — Lab 3:** skills that call other skills as a graph, and a skill library that learns, Voyager-style